# 59 · Does the order of switch-on positions inside mouse S2 replicate in external mice?

Notebook 58 found that the two control mice agree on where genes switch on inside S2 beyond what
their three segment means say (partial Spearman given the switch fraction r). Both mice share one
coordinate, built on one embedding, so that agreement could still come from the coordinate. This
notebook asks the same question of **external mice with an external coordinate**: single-nucleus
PT profiles from adult male mice in CELLxGENE Census (dataset 25818bf7), with a diffusion
pseudotime built **per donor** from genes disjoint from the tested genes.

Our onsets come from notebook 58's outputs for each coordinate that has been run (SCF13 primary;
DPT13, SCF13-ED and later coordinates as sensitivities). The external side does not depend on our
coordinate. The protocol was frozen in workstream B's phase-1 plan (section 8).

Human is not tested here: Lake 2023 merges PT-S1 and PT-S2, and the Census human cortex dataset
labels only convoluted PT and S3, so no external human data place a switch inside S2.

## 1 · Folders and pre-specified rules

| Rule | Value |
|---|---|
| External donors | Census mouse PT nuclei, male, 3–6 months and 12 months (21-day, 21-month and embryonic stages excluded) |
| External coordinate | per donor: log1p(count / total × 10⁴); 2,000 seurat-flavour HVGs **excluding** every gene notebook 58 tested for onset order on any coordinate and the six orientation markers; PCA 30; 30-neighbour graph; diffusion pseudotime rooted at the nucleus with the highest early-marker score (Slc5a2, Slc5a12, Gatm against Slc22a7, Slc7a13, Cyp7b1); scaled to [0, 1] |
| Donor check | S1 < S2 < S3 medians on the donor's pseudotime, else the donor is excluded |
| External onsets | notebook 58's descriptors per donor within its 1st–99th percentile range: curves, logistic transitions, segment units, monotone-like rule, half-amplitude onset, r |
| External onset of a gene | median over donors in which it is monotone-like with our direction; needs ≥ 3 donors |
| **Primary metric** | partial Spearman of our mouse onset (mean of the two mice, notebook 58 primary set: fold-protected, S2 onsets in both mice) with the external onset, given our r (mean of the two mice); 2,000 donor bootstraps (seed 59) |
| **Decision** | claim if the SCF13 partial ρ ≥ 0.2 with the donor-bootstrap interval excluding 0 · abandon framing F2(a) if it is < 0.1 here and notebook 58's primary is < 0.1 |

In [ ]:
import argparse
import json
import os
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
census_file = data_root / 'external' / 'census_pt_cells' / 'mouse_pt_cells.h5ad'
nb58_root = results_root / 'pt_pathway_continuous' / '58_descriptors'
output = results_root / 'pt_pathway_continuous' / '59_external_onsets'
tables = output / 'tables'
tables.mkdir(parents=True, exist_ok=True)
for path in (census_file, nb58_root / 'scf13' / 'tables' / 'iv_primary_mouse_s2_onsets.csv'):
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path}')

NOTEBOOK_LOGIC_VERSION = '59.external_onsets.3'  # Bump when a cached calculation changes.
ADULT_STAGES = ('3-6 month-old mature stage', '12-month-old stage')
SEGMENT_MAP = {'epithelial cell of proximal tubule segment 1': 'PT-S1',
               'epithelial cell of proximal tubule segment 2': 'PT-S2',
               'epithelial cell of proximal tubule segment 3': 'PT-S3'}
EARLY, LATE = ('Slc5a2', 'Slc5a12', 'Gatm'), ('Slc22a7', 'Slc7a13', 'Cyp7b1')
N_HVG, N_PCS, N_NEIGHBORS, SEED = 2000, 30, 30, 59
BASIS_DF, MIN_AMPLITUDE, MIN_ABS_RHO, MIN_RANGE = 6, .5, .9, .1   # notebook 58
MIN_DONORS, N_BOOT = 3, 2000
CLAIM, ABANDON = .2, .1
print('Results folder:', output)

## 2 · External nuclei and the genes to protect

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display
from scipy import sparse
from scipy.stats import spearmanr

from pseudospace import continuous_descriptors as descriptors_module
from pseudospace.continuous_descriptors import (logistic_transitions, monotone_onsets, partial_spearman,
                                                segment_means, segment_units, specimen_curves, switch_fraction,
                                                within_segment_slopes)
from pseudospace.stage_cache import cached_payload, digest
from pseudospace.stats_gam import gam_internal_knots

ours = {}
for folder in sorted(nb58_root.glob('*/tables/iv_primary_mouse_s2_onsets.csv')):
    ours[folder.parents[1].name] = pd.read_csv(folder, index_col=0)
print('Coordinates with notebook 58 onsets:', {k: len(v) for k, v in ours.items()})
protected = set().union(*[set(v.index) for v in ours.values()]) | set(EARLY) | set(LATE)

census = ad.read_h5ad(census_file)
obs = census.obs
keep = (obs.sex.astype(str).eq('male') & obs.development_stage.astype(str).isin(ADULT_STAGES)
        & obs.cell_type.astype(str).isin(SEGMENT_MAP)).to_numpy()
census = census[keep].copy()
census.obs['segment'] = census.obs.cell_type.astype(str).map(SEGMENT_MAP)
donors = sorted(census.obs.donor_id.astype(str).unique())
display(census.obs.groupby(['donor_id', 'development_stage', 'segment'], observed=True).size().unstack())
cache = output / 'stage_cache'
code = digest([NOTEBOOK_LOGIC_VERSION, Path(descriptors_module.__file__)])

## 3 · Per-donor pseudotime and descriptors

In [ ]:


def donor_descriptors(donor):
    sub = census[census.obs.donor_id.astype(str).eq(donor).to_numpy()].copy()
    counts = sparse.csr_matrix(sub.X, dtype=float)
    library = np.asarray(counts.sum(axis=1)).ravel()
    lognorm = counts.multiply((1e4 / np.maximum(library, 1))[:, None]).tocsr()
    lognorm.data = np.log1p(lognorm.data)
    work = ad.AnnData(X=lognorm, obs=sub.obs[['segment']].copy(), var=pd.DataFrame(index=sub.var_names))
    candidates = ~work.var_names.isin(sorted(protected))
    hv = sc.pp.highly_variable_genes(work[:, candidates].copy(), flavor='seurat', n_top_genes=N_HVG, inplace=False)
    hvg = work.var_names[candidates][hv.highly_variable.to_numpy()]
    emb = work[:, hvg].copy()
    sc.pp.pca(emb, n_comps=N_PCS, random_state=SEED)
    sc.pp.neighbors(emb, n_neighbors=N_NEIGHBORS, use_rep='X_pca', random_state=SEED)
    sc.tl.diffmap(emb, random_state=SEED)
    markers = work[:, list(EARLY + LATE)].X.toarray()
    z = (markers - markers.mean(axis=0)) / np.maximum(markers.std(axis=0), 1e-6)
    early_score = z[:, :3].mean(axis=1) - z[:, 3:].mean(axis=1)
    emb.uns['iroot'] = int(np.argmax(early_score))
    sc.tl.dpt(emb)
    t = emb.obs.dpt_pseudotime.to_numpy(float)
    finite = np.isfinite(t)
    t = (t - t[finite].min()) / np.ptp(t[finite])
    seg = work.obs.segment.astype(str).to_numpy()
    medians = pd.Series(t[finite]).groupby(seg[finite]).median()
    ordered = bool(medians['PT-S1'] < medians['PT-S2'] < medians['PT-S3'])
    lo, hi = np.quantile(t[finite], [.01, .99])
    rows = finite & (t >= lo) & (t <= hi)
    out = {'ordered': np.array(ordered), 'medians': medians.reindex(['PT-S1', 'PT-S2', 'PT-S3']).to_numpy(float),
           'n_nuclei': np.array(int(rows.sum())), 'hvg_overlap_protected': np.array(len(set(hvg) & protected))}
    if not ordered:
        return out
    grid = np.linspace(lo, hi, 61)
    position, specimen = t[rows], np.repeat(donor, rows.sum())
    curves = specimen_curves(lognorm[rows], position, specimen, grid, gam_internal_knots(position, basis_df=BASIS_DF))[donor]
    transitions = logistic_transitions(position, seg[rows], specimen).set_index('transition').position
    out['transitions'] = transitions.reindex(['S1→S2', 'S2→S3']).to_numpy(float)
    if not grid[0] < out['transitions'][0] < out['transitions'][1] < grid[-1]:
        out['ordered'] = np.array(False)
        return out
    units = segment_units(grid, *out['transitions'], grid[0], grid[-1])
    onsets = monotone_onsets(curves, units, min_amplitude=MIN_AMPLITUDE, min_abs_rho=MIN_ABS_RHO)
    out['slope_t'] = within_segment_slopes(lognorm[rows], position, specimen, seg[rows])[donor].to_numpy(float)
    out.update({'onset': onsets.onset.to_numpy(float), 'direction': onsets.direction.to_numpy(float),
                'monotone': onsets.monotone.to_numpy(float),
                'r': switch_fraction(segment_means(lognorm[rows], seg[rows]), MIN_RANGE),
                'means': segment_means(lognorm[rows], seg[rows])})
    return out


external = {d: cached_payload('donor_' + d, lambda d=d: donor_descriptors(d), root=cache,
                              params={'hvg': N_HVG, 'pcs': N_PCS, 'neighbors': N_NEIGHBORS, 'seed': SEED,
                                      'rules': [MIN_AMPLITUDE, MIN_ABS_RHO, MIN_RANGE], 'protected': sorted(protected)},
                              inputs={'file': [census_file.name, census_file.stat().st_size]}, code=code)
            for d in donors}
donor_table = pd.DataFrame({d: {'nuclei in range': int(v['n_nuclei']), 'S1 < S2 < S3': bool(v['ordered']),
                                'median S1 / S2 / S3': ' / '.join(f'{x:.2f}' for x in v['medians']),
                                'S1→S2, S2→S3': ' / '.join(f'{x:.2f}' for x in v.get('transitions', [np.nan, np.nan])),
                                'monotone-like genes': int(np.nansum(v.get('monotone', [0]))),
                                'protected genes among HVGs (must be 0)': int(v['hvg_overlap_protected'])}
                            for d, v in external.items()}).T
display(donor_table)
if donor_table['protected genes among HVGs (must be 0)'].sum():
    raise ValueError('A protected gene entered an external coordinate.')
donor_table.to_csv(tables / 'external_donors.csv')
used = [d for d, v in external.items() if bool(v['ordered'])]
gene_names = pd.Index(census.var_names)
onset_matrix = pd.DataFrame({d: external[d]['onset'] for d in used}, index=gene_names)
direction_matrix = pd.DataFrame({d: external[d]['direction'] for d in used}, index=gene_names)
monotone_matrix = pd.DataFrame({d: external[d]['monotone'].astype(bool) for d in used}, index=gene_names)
r_matrix = pd.DataFrame({d: external[d]['r'] for d in used}, index=gene_names)
print(f'{len(used)} of {len(donors)} donors used')

## 4 · Our onsets against external onsets, beyond our r

In [ ]:


def external_onsets(direction, donor_columns):
    genes_ = direction.index
    valid = (monotone_matrix.loc[genes_, donor_columns].to_numpy()
             & np.equal(direction_matrix.loc[genes_, donor_columns].to_numpy(), direction.to_numpy()[:, None]))
    values = np.where(valid, onset_matrix.loc[genes_, donor_columns].to_numpy(), np.nan)
    count = valid.sum(axis=1)
    with np.errstate(all='ignore'):
        median = np.array([np.median(v[np.isfinite(v)]) if c >= MIN_DONORS else np.nan for v, c in zip(values, count)])
    return pd.Series(median, index=genes_)


rows, matched = [], {}
rng = np.random.default_rng(SEED)
boot_donors = [list(rng.choice(used, len(used), replace=True)) for _ in range(N_BOOT)]
for label, table in ours.items():
    t = table[table.index.isin(gene_names)].copy()
    our_onset = (t.onset_a + t.onset_b) / 2
    our_r = (t.r_a + t.r_b) / 2
    direction = pd.Series(t.dir_a.to_numpy(), index=t.index)
    sub = direction.reindex(gene_names).dropna()
    ext = external_onsets(sub, used).reindex(t.index)
    ext_r = r_matrix[used].median(axis=1).reindex(t.index)
    m = ext.notna() & our_r.notna()
    frame = pd.DataFrame({'our onset': our_onset, 'our r': our_r, 'external onset': ext, 'external r': ext_r})[m]
    frame.to_csv(tables / f'onsets_{label}.csv')
    matched[label] = frame
    row = {'coordinate': label, 'genes in our set': len(t), 'genes with external onset': int(m.sum())}
    if m.sum() >= 20:
        estimate = partial_spearman(frame['our onset'], frame['external onset'], frame[['our r']].to_numpy())
        ok = frame['external r'].notna()
        both = partial_spearman(frame['our onset'][ok], frame['external onset'][ok],
                                frame.loc[ok, ['our r', 'external r']].to_numpy()) if ok.sum() >= 20 else np.nan
        draws = []
        for chosen in boot_donors:
            e = external_onsets(sub, chosen).reindex(frame.index)
            ok = e.notna()
            draws.append(partial_spearman(frame['our onset'][ok], e[ok], frame[['our r']][ok].to_numpy())
                         if ok.sum() >= 20 else np.nan)
        draws = np.array(draws)
        row.update({'partial Spearman (external | our r)': estimate,
                    '95% low (donor bootstrap)': float(np.nanquantile(draws, .025)),
                    '95% high (donor bootstrap)': float(np.nanquantile(draws, .975)),
                    'partial Spearman given our and external r': both,
                    'Spearman our onset vs external onset': spearmanr(frame['our onset'], frame['external onset']).statistic,
                    'Spearman our r vs external r': spearmanr(frame['our r'], frame['external r'], nan_policy='omit').statistic})
    rows.append(row)
REPLICATION = pd.DataFrame(rows)
REPLICATION.to_csv(tables / 'external_onset_replication.csv', index=False)
display(REPLICATION.round(3))

## 5 · Decision

In [ ]:
primary = REPLICATION.set_index('coordinate').loc['scf13'] if 'scf13' in set(REPLICATION.coordinate) else None
estimate = primary.get('partial Spearman (external | our r)', np.nan) if primary is not None else np.nan
low = primary.get('95% low (donor bootstrap)', np.nan) if primary is not None else np.nan
nb58_primary = pd.read_csv(nb58_root / 'scf13' / 'tables' / 'iv_decision.csv', index_col=0).value
nb58_value = float(nb58_primary['primary partial ρ (fold-protected, mouse S2)'])
if estimate >= CLAIM and low > 0:
    verdict = 'claim: the within-S2 onset order replicates in external mice beyond segment means'
elif estimate < ABANDON and nb58_value < ABANDON:
    verdict = 'abandon framing F2(a)'
else:
    verdict = 'not replicated externally at the pre-specified level'
decision = pd.Series({'SCF13 external partial ρ': estimate, 'donor-bootstrap 95% low': low,
                      'notebook 58 primary partial ρ': nb58_value, 'donors used': len(used), 'decision': verdict},
                     name='value')
decision.to_csv(tables / 'decision.csv')
display(decision.to_frame())
(output / 'run_record.json').write_text(json.dumps({
    'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'donors_used': used,
    'replication': REPLICATION.to_dict(orient='records'),
    'decision': {k: (v if isinstance(v, str) else float(v)) for k, v in decision.items()}}, indent=2, default=float))

## 6 · Post hoc: do within-segment gradients agree with external mice?

**Added after notebook 58 showed that slopes inside coordinate-threshold segments replicate between
our specimens beyond both step nulls, and after this notebook's onset result was seen.** Between-specimen
replication cannot tell anatomical position from any other axis shared by our specimens. Here each
gene's within-segment slope t along the external donor's own pseudotime (median over donors) is
compared with our mouse slopes (mean of the two mice) over fold-protected genes, separately for
slopes inside label segments and inside coordinate-threshold segments. Donor bootstrap as above (a
median over resampled donors is noisier than over all six, so the interval sits below the estimate).
Label noise in both datasets lets segment steps leak into within-segment slopes, so the agreement is
also given as a partial correlation after removing the external S2 − S1 and S3 − S2 segment steps.
No decision rule was set in advance; the numbers are reported as they are.

In [ ]:
external_slopes = {seg: pd.DataFrame({d: external[d]['slope_t'][:, k] for d in used}, index=gene_names)
                   for k, seg in enumerate(('S1', 'S2', 'S3'))}
means_by_donor = {d: external[d]['means'] for d in used}
contrasts = {'S2-S1': pd.DataFrame({d: m[:, 1] - m[:, 0] for d, m in means_by_donor.items()}, index=gene_names).median(axis=1),
             'S3-S2': pd.DataFrame({d: m[:, 2] - m[:, 1] for d, m in means_by_donor.items()}, index=gene_names).median(axis=1)}
slope_rows = []
for label in ours:
    path = nb58_root / label / 'tables' / 'i_slopes_per_gene.csv'
    if not path.is_file():
        continue
    slopes = pd.read_csv(path, index_col=0)
    slopes = slopes[slopes.fold_protected.astype(bool) & slopes.index.isin(gene_names)]
    for seg in ('S1', 'S2', 'S3'):
        ext_med = external_slopes[seg].reindex(slopes.index).median(axis=1)
        for kind in ('label', 'coordinate'):
            ours_t = slopes[[f'{m} {seg} {kind}' for m in ('Ctrl1A2', 'Ctrl1A4')]].mean(axis=1)
            ok = ours_t.notna() & ext_med.notna()
            estimate = spearmanr(ours_t[ok], ext_med[ok]).statistic
            draws = []
            for chosen in boot_donors[:500]:
                e = external_slopes[seg].reindex(slopes.index)[chosen].median(axis=1)
                k_ = ok & e.notna()
                draws.append(spearmanr(ours_t[k_], e[k_]).statistic)
            # Shared step structure leaks into within-label slopes in both datasets; remove it.
            steps = pd.DataFrame({'ext S2-S1': contrasts['S2-S1'].reindex(slopes.index),
                                  'ext S3-S2': contrasts['S3-S2'].reindex(slopes.index)})
            okp = ok & steps.notna().all(axis=1)
            beyond = partial_spearman(ours_t[okp], ext_med[okp], steps[okp].to_numpy())
            slope_rows.append({'coordinate': label, 'segment': seg, 'our slopes inside': kind, 'genes': int(ok.sum()),
                               'Spearman with external slopes': estimate,
                               'partial Spearman given external segment steps': beyond,
                               '95% low (donor bootstrap)': float(np.nanquantile(draws, .025)),
                               '95% high (donor bootstrap)': float(np.nanquantile(draws, .975))})
EXTERNAL_SLOPES = pd.DataFrame(slope_rows)
EXTERNAL_SLOPES.to_csv(tables / 'posthoc_external_slope_agreement.csv', index=False)
display(EXTERNAL_SLOPES.round(3))